In [0]:
%pip install langchain pdfminer.six python-docx openpyxl pandas gpt4all faiss-cpu unstructured


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/981.5 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.8 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.8 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [0]:
 %restart_python

In [0]:
import os
import pandas as pd
from langchain.document_loaders import (
    PDFMinerLoader, Docx2txtLoader, CSVLoader, TextLoader, UnstructuredFileLoader
)
from langchain.embeddings import GPT4AllEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import GPT4All

# Load any document based on its extension
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".pdf":
            return PDFMinerLoader(file_path).load()
        elif ext == ".docx":
            return Docx2txtLoader(file_path).load()
        elif ext == ".csv":
            return CSVLoader(file_path).load()
        elif ext == ".txt":
            return TextLoader(file_path).load()
        elif ext in [".xlsx", ".xls"]:
            df = pd.read_excel(file_path)
            temp_path = "/Volumes/workspace/default/adarsh_files/Adarsh.pdf"
            df.to_string(open(temp_path, "w", encoding="utf-8"))
            return TextLoader(temp_path).load()
        else:
            return UnstructuredFileLoader(file_path).load()
    except Exception as e:
        print(f"Failed to load {file_path}: {e}")
        return []

# Load all documents from a DBFS folder
def load_documents_from_folder(folder_path):
    docs = []
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)
        if os.path.isfile(full_path):
            print(f"Loading: {file}")
            docs += load_file(full_path)
    return docs

# Create a QA chain with local LLM
def build_qa_chain(docs, model_path):
    embeddings = GPT4AllEmbeddings()
    vectorstore = FAISS.from_documents(docs, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    llm = GPT4All(model=model_path, verbose=False)
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=False
    )
    return chain

# Set paths
docs_folder = "/dbfs/FileStore/uploads/docs/"  # Change if needed
model_path = "/dbfs/FileStore/models/ggml-gpt4all-j-v1.3-groovy.bin"  # Your local LLM

# Run setup
documents = load_documents_from_folder(docs_folder)
qa_chain = build_qa_chain(documents, model_path)

print("System ready. You can now ask any question about the uploaded documents.")


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-4505251183794523>, line 2
      1 import os
----> 2 import pandas as pd
      3 from langchain.document_loaders import (
      4     PDFMinerLoader, Docx2txtLoader, CSVLoader, TextLoader, UnstructuredFileLoader
      5 )
      6 from langchain.embeddings import GPT4AllEmbeddings

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/hook.py:71, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     65 if not self._should_hint and (
     66     (module := sys.modules.get(absolute_name)) is not None and
     67     (fname := get_allowed_file_name_or_none(module)) is not None and
     68     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     69         absolute_name, float("inf")) and not self._should_hint):
     70     self._should_hint = True
---> 71 module

In [0]:
question = "What is this document about?"
print("Answer:", qa_chain.run(question))
